# CO × Lyα: aligned periodic maps
Install this repository with `python -m pip install -e .` from its root first.
Requires h5py and matplotlib for loading/plotting. The Lyα input is **delta_F**,
not flux. Verify that both maps describe the same redshift, origin, voxel
centers, axis order and redshift-space convention before interpreting results.
This is a noiseless-field, diagonal Gaussian S/N baseline, not a HIPSTER sparse
sightline estimator or a calibrated reconstruction forecast.


In [ ]:
from pathlib import Path
import numpy as np
import h5py
import matplotlib.pyplot as plt
from lya_co.co.co_ly_sn import rebin_mean, combine_maps_and_sn

CO_FILE = Path('/users/jiawenfan/work/lya_CO/limlam_cartesian_v2_simplified/co_li_cartesian.npz')
LYA_FILE = Path('/projects/pbreysse/lya_forest_CO/lya_storage/Lyman-alpha-ASTRID_z2.5_unzipped/Lyman-alpha-ASTRID_z2.5/map_ASTRID_true_0.25_z2.5.hdf5')
BOX = np.array([250., 250., 250.])  # cMpc/h, in final x,y,z order
K_EDGES = np.linspace(0.03, 1.0, 21)
NOISE_CO = 0.0  # microK^2 (cMpc/h)^3; zero = sample variance only
NOISE_F = 0.0   # (cMpc/h)^3


In [ ]:
with np.load(CO_FILE, allow_pickle=False) as data:
    Tco = np.asarray(data['map_cube'])
with h5py.File(LYA_FILE, 'r') as data:
    print('Lyα metadata:', dict(data.attrs), dict(data['map'].attrs))
    # Read/average slabs so the full 1000^3 map is not loaded at once.
    # Assumes input x,y,z already matches CO. No guessed axis swap.
    delta_f = rebin_mean(data['map'], Tco.shape)
print('Matched shape:', Tco.shape, 'voxel size [cMpc/h]:', BOX / Tco.shape)
if K_EDGES[-1] > 0.5 * np.min(np.pi*np.asarray(Tco.shape)/BOX):
    raise ValueError('Reduce kmax or increase map resolution')
result = combine_maps_and_sn(Tco, delta_f, BOX, K_EDGES, NOISE_CO, NOISE_F)
print('Cross S/N (diagonal Gaussian):', result['sn_cross_total'])


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
k = result['k']
axes[0].plot(k, result['P_CO_F'])
axes[0].axhline(0, color='gray', lw=0.8)
axes[0].set_ylabel(r'$P_{CO,F}\ [\mu K\,(\mathrm{cMpc}/h)^3]$')
axes[1].plot(k, result['r'])
axes[1].set_ylabel('Correlation coefficient r')
axes[1].set_ylim(-1.05, 1.05)
axes[2].plot(k, result['sn_cross'])
axes[2].set_ylabel('Cross S/N per bin')
for ax in axes:
    ax.set_xlabel(r'$k\ [h/\mathrm{cMpc}]$')
fig.tight_layout()
plt.show()


In [ ]:
output = Path('co_lya_sn_results.npz')
# Exclusive creation avoids silently replacing an earlier result.
with output.open('xb') as handle:
    np.savez(handle, **result, co_file=str(CO_FILE), lya_file=str(LYA_FILE),
             lya_quantity='delta_F', geometry='periodic aligned Cartesian',
             covariance='diagonal Gaussian', noise_model='user supplied')
